# 🫀 Pipeline Pengenalan Pola — Heart Disease Classification

> **Mata Kuliah: Pengenalan Pola**  
> Pipeline lengkap: Data Dummy → EDA → Cleaning → Normalisasi → Feature Selection → Bayesian Decision → Evaluasi

---

## 📌 Tujuan Pembelajaran
1. Memahami pipeline pengenalan pola end-to-end secara **manual** (tanpa sklearn untuk model)
2. Menangani dataset **extremely imbalanced** (ketidakseimbangan kelas ekstrem)
3. Melakukan **feature selection** berdasarkan statistik
4. Membangun **Bayesian Decision Classifier** dari nol
5. Mengevaluasi dengan **Precision, Recall, F1-Score**

---

## 🗂️ Dataset
- **20 data dummy** terinspirasi Heart Disease UCI
- **4 fitur**: `age`, `cholesterol`, `max_heart_rate`, `blood_pressure`
- **1 label**: `disease` (0 = Sehat, 1 = Sakit)
- **Extremely Imbalanced**: 17 sehat (85%), 3 sakit (15%)

---
# 📦 STEP 0 — Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Agar plot tampil inline
%matplotlib inline
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'

print('✅ Library berhasil diimport!')
print(f'   NumPy  : {np.__version__}')
print(f'   Pandas : {pd.__version__}')

---
# 📊 STEP 1 — Membuat Dataset Dummy

Dataset dibuat secara **manual** agar kita bisa mengontrol distribusi fitur dan memastikan **extremely imbalanced**.

| Fitur | Keterangan | Satuan |
|---|---|---|
| `age` | Usia pasien | tahun |
| `cholesterol` | Kadar kolesterol | mg/dL |
| `max_heart_rate` | Detak jantung maksimum | bpm |
| `blood_pressure` | Tekanan darah sistolik | mmHg |
| `disease` | Label: 0=Sehat, 1=Sakit | — |

In [ ]:
# ============================================================
# DATASET DUMMY — 20 data, 4 fitur, extremely imbalanced
# Kelas 0 (Sehat) : 17 data  → 85%
# Kelas 1 (Sakit) :  3 data  → 15%
# ============================================================

np.random.seed(42)

data = {
    'age':            [45, 52, 38, 60, 47, 55, 42, 50, 35, 63,
                       48, 57, 40, 53, 44, 61, 39, 68, 72, 65],
    'cholesterol':    [200, 215, 180, 230, 195, 210, 185, 220, 170, 240,
                       205, 225, 190, 235, 198, 245, 178, 290, 310, 285],
    'max_heart_rate': [150, 145, 160, 130, 155, 140, 162, 148, 165, 125,
                       152, 138, 158, 132, 156, 128, 163, 110, 105, 108],
    'blood_pressure': [120, 125, 115, 135, 122, 130, 118, 128, 112, 140,
                       124, 133, 119, 136, 121, 142, 116, 160, 170, 158],
    'disease':        [  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
                         0,   0,   0,   0,   0,   0,   0,   1,   1,   1]
    # ⚠️  Hanya 3 dari 20 data yang sakit → extremely imbalanced!
}

# Tambahkan 1 missing value secara sengaja untuk latihan cleaning
data['cholesterol'][3] = np.nan   # baris ke-4, kolesterol hilang
data['max_heart_rate'][10] = np.nan  # baris ke-11, heart rate hilang

df = pd.DataFrame(data)

print('=' * 55)
print('         DATASET HEART DISEASE DUMMY')
print('=' * 55)
print(df.to_string())
print('\n📐 Shape :', df.shape)
print('🔢 Tipe  :')
print(df.dtypes)

---
# 🔍 STEP 2 — Exploratory Data Analysis (EDA)

Sebelum memproses data, kita **eksplorasi** terlebih dahulu untuk memahami:
- Distribusi kelas (class imbalance)
- Statistik deskriptif
- Missing values
- Distribusi setiap fitur

In [ ]:
# ============================================================
# 2.1 — Statistik Deskriptif
# ============================================================
print('📊 STATISTIK DESKRIPTIF')
print('=' * 55)
print(df.describe().round(2).to_string())

In [ ]:
# ============================================================
# 2.2 — Cek Missing Values
# ============================================================
print('❓ MISSING VALUES')
print('=' * 40)
mv = df.isnull().sum()
pct = (df.isnull().sum() / len(df) * 100).round(1)
mv_df = pd.DataFrame({'Count': mv, 'Persen (%)': pct})
print(mv_df)
print(f'\n⚠️  Total missing: {df.isnull().sum().sum()} sel dari {df.size} sel')

In [ ]:
# ============================================================
# 2.3 — Distribusi Kelas (Class Imbalance Visualization)
# ============================================================
class_counts = df['disease'].value_counts()
labels = ['Sehat (0)', 'Sakit (1)']
colors = ['#4CAF50', '#F44336']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('⚠️  Distribusi Kelas — Extremely Imbalanced Dataset', 
             fontsize=14, fontweight='bold', y=1.02)

# Bar chart
bars = axes[0].bar(labels, class_counts.values, color=colors, edgecolor='black', width=0.5)
axes[0].set_title('Jumlah Data per Kelas', fontweight='bold')
axes[0].set_ylabel('Jumlah Data')
axes[0].set_ylim(0, 22)
for bar, val in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val} data', ha='center', va='bottom', fontweight='bold', fontsize=12)
axes[0].axhline(y=10, color='gray', linestyle='--', alpha=0.5, label='Balanced baseline')
axes[0].legend()

# Pie chart
wedges, texts, autotexts = axes[1].pie(
    class_counts.values, labels=labels, colors=colors,
    autopct='%1.1f%%', startangle=90, explode=(0, 0.1),
    textprops={'fontsize': 11}, wedgeprops={'edgecolor': 'black'}
)
for at in autotexts:
    at.set_fontweight('bold')
    at.set_fontsize(12)
axes[1].set_title('Proporsi Kelas (%)', fontweight='bold')

# Annotation imbalance ratio
ratio = class_counts[0] / class_counts[1]
axes[1].text(0, -1.35, f'Imbalance Ratio = {ratio:.1f}:1  (Sehat:Sakit)',
             ha='center', fontsize=10, color='red', fontweight='bold')

plt.tight_layout()
plt.savefig('01_class_distribution.png', bbox_inches='tight')
plt.show()
print(f'\n💡 Imbalance Ratio: {ratio:.1f}:1 — Kelas mayoritas {ratio:.0f}x lebih banyak!')

In [ ]:
# ============================================================
# 2.4 — Distribusi Setiap Fitur (per kelas)
# ============================================================
features = ['age', 'cholesterol', 'max_heart_rate', 'blood_pressure']
feat_labels = ['Usia (tahun)', 'Kolesterol (mg/dL)', 'Detak Jantung Maks (bpm)', 'Tekanan Darah (mmHg)']
color_map = {0: '#4CAF50', 1: '#F44336'}

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('📊 Distribusi Fitur per Kelas (Sehat vs Sakit)', fontsize=14, fontweight='bold')
axes = axes.flatten()

for i, (feat, label) in enumerate(zip(features, feat_labels)):
    ax = axes[i]
    for cls, clr in color_map.items():
        subset = df[df['disease'] == cls][feat].dropna()
        ax.hist(subset, bins=6, alpha=0.65, color=clr, edgecolor='black',
                label=f'Kelas {cls} ({"Sehat" if cls==0 else "Sakit"})')
        # Tambah garis mean
        ax.axvline(subset.mean(), color=clr, linestyle='--', linewidth=1.8,
                   label=f'Mean {cls}: {subset.mean():.1f}')
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('Frekuensi')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('02_feature_distribution.png', bbox_inches='tight')
plt.show()
print('💡 Amati perbedaan distribusi antar kelas — fitur yang paling berbeda = paling diskriminatif!')

In [ ]:
# ============================================================
# 2.5 — Heatmap Korelasi (manual)
# ============================================================
df_clean_corr = df[features + ['disease']].dropna()
corr_matrix = df_clean_corr.corr()

fig, ax = plt.subplots(figsize=(7, 5))
fig.suptitle('🔗 Heatmap Korelasi Fitur', fontsize=13, fontweight='bold')

col_names = features + ['disease']
n = len(col_names)
im = ax.imshow(corr_matrix.values, cmap='RdYlGn', vmin=-1, vmax=1)

ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(col_names, rotation=30, ha='right', fontsize=9)
ax.set_yticklabels(col_names, fontsize=9)

for i in range(n):
    for j in range(n):
        val = corr_matrix.values[i, j]
        color = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9,
                fontweight='bold', color=color)

plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig('03_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('💡 Nilai mendekati +1/-1 = korelasi kuat. Fitur berkorelasi tinggi dengan disease = kandidat fitur terbaik!')

---
# 🧹 STEP 3 — Data Cleaning (Manual)

## Strategi Handling Missing Values
Karena dataset sangat kecil (20 data), kita **tidak boleh membuang baris** (drop row).
Strategi: **Imputasi dengan median per kelas** agar tidak mencempur distribusi kelas.

In [ ]:
# ============================================================
# 3.1 — Visualisasi posisi missing value
# ============================================================
fig, ax = plt.subplots(figsize=(9, 4))
fig.suptitle('🔴 Peta Missing Values (Merah = Hilang)', fontsize=13, fontweight='bold')

mv_matrix = df[features].isnull().astype(int).values
cmap = plt.cm.colors.ListedColormap(['#E8F5E9', '#F44336'])
im = ax.imshow(mv_matrix.T, aspect='auto', cmap=cmap, vmin=0, vmax=1)

ax.set_yticks(range(len(features)))
ax.set_yticklabels(features, fontsize=10)
ax.set_xticks(range(20))
ax.set_xticklabels([f'D{i+1}' for i in range(20)], fontsize=7, rotation=45)
ax.set_xlabel('Data ke-', fontsize=10)
ax.set_title('Setiap kolom = fitur, setiap baris = data ke-n', fontsize=9)

# Tandai missing
for i in range(len(features)):
    for j in range(20):
        if df[features[i]].iloc[j] != df[features[i]].iloc[j]:  # isnan
            ax.text(j, i, 'NaN', ha='center', va='center', fontsize=7,
                    color='white', fontweight='bold')

patch0 = mpatches.Patch(color='#E8F5E9', label='Ada Nilai')
patch1 = mpatches.Patch(color='#F44336', label='Missing (NaN)')
ax.legend(handles=[patch0, patch1], loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('04_missing_values_map.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 3.2 — Imputasi Manual: Median per Kelas
# ============================================================
df_clean = df.copy()

imputed_log = []  # untuk logging

for feat in features:
    missing_mask = df_clean[feat].isnull()
    if missing_mask.sum() == 0:
        continue
    for idx in df_clean[missing_mask].index:
        kelas = df_clean.loc[idx, 'disease']
        # Hitung median kelas tersebut (exclude missing)
        median_val = df_clean[df_clean['disease'] == kelas][feat].median()
        df_clean.loc[idx, feat] = median_val
        imputed_log.append({
            'Index': idx, 'Fitur': feat,
            'Kelas': kelas, 'Nilai Imputasi': round(median_val, 2)
        })

print('✅ IMPUTASI SELESAI — LOG DETAIL:')
print('=' * 50)
for log in imputed_log:
    print(f"  Baris {log['Index']:>2} | Fitur: {log['Fitur']:<16} | "
          f"Kelas={log['Kelas']} | Imputasi={log['Nilai Imputasi']}")

print(f'\n✅ Sisa missing values: {df_clean.isnull().sum().sum()}')
print('\n📋 Dataset setelah cleaning:')
print(df_clean.to_string())

In [ ]:
# ============================================================
# 3.3 — Visualisasi: Sebelum vs Sesudah Imputasi
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('🧹 Perbandingan Data: Sebelum vs Sesudah Imputasi', fontsize=13, fontweight='bold')

for ax, (data_plot, title) in zip(axes, [(df, 'Sebelum Cleaning (Ada NaN)'), 
                                           (df_clean, 'Sesudah Cleaning (Bersih)')]):
    for cls, clr, mrk in [(0, '#4CAF50', 'o'), (1, '#F44336', '^')]:
        sub = data_plot[data_plot['disease'] == cls]
        ax.scatter(sub['age'], sub['cholesterol'], c=clr, marker=mrk, s=80,
                   edgecolors='black', linewidths=0.7,
                   label=f'Kelas {cls} ({"Sehat" if cls==0 else "Sakit"})')
    ax.set_xlabel('Usia (age)')
    ax.set_ylabel('Kolesterol (cholesterol)')
    ax.set_title(title, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)

# Tandai titik yang diimputasi
for log in imputed_log:
    row = df_clean.iloc[log['Index']]
    axes[1].annotate(f"↑ imputasi", (row['age'], row['cholesterol']),
                     textcoords='offset points', xytext=(5, 5),
                     fontsize=7, color='blue', arrowprops=dict(arrowstyle='->', color='blue'))

plt.tight_layout()
plt.savefig('05_before_after_cleaning.png', bbox_inches='tight')
plt.show()

---
# 📏 STEP 4 — Normalisasi (Min-Max Scaling Manual)

### Mengapa perlu normalisasi?
- Fitur `cholesterol` (170–310) vs `blood_pressure` (112–170) memiliki skala berbeda
- Bayesian Classifier menghitung **probabilitas** dari distribusi — skala besar bisa mendominasi variance
- **Min-Max Scaling** → semua fitur masuk rentang [0, 1]

$$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

In [ ]:
# ============================================================
# 4.1 — Min-Max Normalization Manual
# ============================================================
X = df_clean[features].values.astype(float)
y = df_clean['disease'].values

# Simpan parameter scaler (harus dari training set, di sini = all data karena kecil)
x_min = X.min(axis=0)
x_max = X.max(axis=0)

X_norm = (X - x_min) / (x_max - x_min)

df_norm = pd.DataFrame(X_norm, columns=features)
df_norm['disease'] = y

print('📏 PARAMETER NORMALISASI (Min-Max per Fitur):')
print('=' * 55)
for i, feat in enumerate(features):
    print(f'  {feat:<20} | min={x_min[i]:.1f}  max={x_max[i]:.1f}')

print('\n✅ Data setelah normalisasi (5 baris pertama):')
print(df_norm.head().round(4).to_string())

In [ ]:
# ============================================================
# 4.2 — Visualisasi Sebelum vs Sesudah Normalisasi (Boxplot)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('📏 Distribusi Fitur: Sebelum vs Sesudah Normalisasi', fontsize=13, fontweight='bold')

# Sebelum
bp1 = axes[0].boxplot(
    [df_clean[f].values for f in features],
    labels=features, patch_artist=True,
    boxprops=dict(facecolor='#BBDEFB'), medianprops=dict(color='red', linewidth=2)
)
axes[0].set_title('Sebelum Normalisasi', fontweight='bold')
axes[0].set_ylabel('Nilai Asli')
axes[0].tick_params(axis='x', rotation=20)
axes[0].grid(axis='y', alpha=0.3)

# Sesudah
bp2 = axes[1].boxplot(
    [df_norm[f].values for f in features],
    labels=features, patch_artist=True,
    boxprops=dict(facecolor='#C8E6C9'), medianprops=dict(color='red', linewidth=2)
)
axes[1].set_title('Sesudah Normalisasi [0, 1]', fontweight='bold')
axes[1].set_ylabel('Nilai Ternormalisasi')
axes[1].set_ylim(-0.1, 1.1)
axes[1].tick_params(axis='x', rotation=20)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('06_normalization_comparison.png', bbox_inches='tight')
plt.show()
print('💡 Setelah normalisasi, semua fitur berada di skala [0,1] — adil untuk perbandingan!')

---
# 🎯 STEP 5 — Feature Selection (Manual)

Kita pilih **1–3 fitur terbaik** menggunakan 2 metode statistik sederhana:

### Metode 1: Fisher's Score (Separability)
$$Fisher_i = \frac{(\mu_1 - \mu_0)^2}{\sigma_1^2 + \sigma_0^2}$$
Semakin besar → semakin baik memisahkan kelas.

### Metode 2: Korelasi dengan Label
$$|\rho_{fitur, label}|$$
Semakin tinggi korelasinya dengan label → semakin relevan.

In [ ]:
# ============================================================
# 5.1 — Hitung Fisher's Score Manual
# ============================================================
fisher_scores = {}
correlation_scores = {}

print('🎯 FEATURE SELECTION — Skor per Fitur')
print('=' * 60)
print(f'{"Fitur":<22} {"μ Sehat":>8} {"μ Sakit":>8} {"Fisher":>10} {"Korelasi":>10}')
print('-' * 60)

for i, feat in enumerate(features):
    x0 = X_norm[y == 0, i]  # kelas sehat
    x1 = X_norm[y == 1, i]  # kelas sakit

    mu0, mu1 = x0.mean(), x1.mean()
    s0, s1   = x0.var(),  x1.var()

    # Fisher's Score
    denom = s0 + s1 + 1e-10  # hindari div/0
    fisher = (mu1 - mu0) ** 2 / denom
    fisher_scores[feat] = fisher

    # Korelasi Pearson manual
    xi = X_norm[:, i]
    corr = abs(np.corrcoef(xi, y)[0, 1])
    correlation_scores[feat] = corr

    print(f'{feat:<22} {mu0:>8.3f} {mu1:>8.3f} {fisher:>10.4f} {corr:>10.4f}')

print('=' * 60)

In [ ]:
# ============================================================
# 5.2 — Visualisasi Fisher Score & Korelasi
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('🎯 Feature Selection — Skor Diskriminasi Fitur', fontsize=13, fontweight='bold')

# Fisher Score
sorted_fisher = sorted(fisher_scores.items(), key=lambda x: x[1], reverse=True)
f_names, f_vals = zip(*sorted_fisher)
bar_colors = ['#F44336' if i < 3 else '#90A4AE' for i in range(len(f_names))]
bars = axes[0].barh(f_names, f_vals, color=bar_colors, edgecolor='black')
axes[0].set_title("Fisher's Score\n(lebih besar = lebih diskriminatif)", fontweight='bold')
axes[0].set_xlabel("Fisher's Score")
axes[0].invert_yaxis()
for bar, val in zip(bars, f_vals):
    axes[0].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)
axes[0].axvline(x=0, color='black', linewidth=0.8)

# Korelasi
sorted_corr = sorted(correlation_scores.items(), key=lambda x: x[1], reverse=True)
c_names, c_vals = zip(*sorted_corr)
bar_colors2 = ['#2196F3' if i < 3 else '#90A4AE' for i in range(len(c_names))]
bars2 = axes[1].barh(c_names, c_vals, color=bar_colors2, edgecolor='black')
axes[1].set_title('Korelasi |Pearson| dengan Label\n(lebih besar = lebih relevan)', fontweight='bold')
axes[1].set_xlabel('|Korelasi|')
axes[1].set_xlim(0, 1.1)
axes[1].invert_yaxis()
for bar, val in zip(bars2, c_vals):
    axes[1].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9, fontweight='bold')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('07_feature_selection.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 5.3 — Pilih Top-3 Fitur (rata-rata rank)
# ============================================================

# Ranking berdasarkan Fisher (rank 1 = terbaik)
fisher_rank = {k: r+1 for r, (k, _) in enumerate(
    sorted(fisher_scores.items(), key=lambda x: x[1], reverse=True))}

# Ranking berdasarkan Korelasi
corr_rank = {k: r+1 for r, (k, _) in enumerate(
    sorted(correlation_scores.items(), key=lambda x: x[1], reverse=True))}

# Rata-rata rank
avg_rank = {feat: (fisher_rank[feat] + corr_rank[feat]) / 2 for feat in features}

print('🏆 RANKING FITUR (rata-rata rank Fisher + Korelasi):')
print('=' * 55)
print(f'{"Fitur":<22} {"Rank Fisher":>12} {"Rank Korel":>10} {"Rata-rata":>10}')
print('-' * 55)
sorted_avg = sorted(avg_rank.items(), key=lambda x: x[1])
for feat, avg in sorted_avg:
    marker = '  ⭐ PILIH' if avg <= 2.5 else ''
    print(f'{feat:<22} {fisher_rank[feat]:>12} {corr_rank[feat]:>10} {avg:>10.1f}{marker}')

# Pilih top-3
top3_features = [feat for feat, _ in sorted_avg[:3]]
print(f'\n✅ Fitur Terpilih (Top-3): {top3_features}')

# Update X untuk modeling
feat_idx = [features.index(f) for f in top3_features]
X_selected = X_norm[:, feat_idx]
print(f'📐 Shape X_selected: {X_selected.shape}')

In [ ]:
# ============================================================
# 5.4 — Scatter Plot: Fitur Terpilih vs Label
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('🔍 Visualisasi Fitur Terpilih vs Label', fontsize=13, fontweight='bold')

for ax, feat in zip(axes, top3_features):
    for cls, clr, mrk, lbl in [(0, '#4CAF50', 'o', 'Sehat'), (1, '#F44336', '^', 'Sakit')]:
        mask = y == cls
        fidx = features.index(feat)
        ax.scatter(np.where(mask)[0], X_norm[mask, fidx],
                   c=clr, marker=mrk, s=100, edgecolors='black', label=lbl, zorder=3)
    ax.set_title(f'{feat}\n(ternormalisasi)', fontweight='bold')
    ax.set_xlabel('Index Data')
    ax.set_ylabel('Nilai [0,1]')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.savefig('08_selected_features_scatter.png', bbox_inches='tight')
plt.show()
print('💡 Perhatikan cluster data Sakit (segitiga merah) — apakah terpisah dari Sehat (lingkaran hijau)?')

---
# 🧠 STEP 6 — Bayesian Decision Classifier (Manual)

## Teori: Bayesian Decision Theory

Klasifikasi berdasarkan **Teorema Bayes**:

$$P(C_k | \mathbf{x}) \propto P(\mathbf{x} | C_k) \cdot P(C_k)$$

- $P(C_k)$ = **Prior probability** kelas $k$ (dari data latih)
- $P(\mathbf{x}|C_k)$ = **Likelihood** = asumsi Gaussian per fitur
- $P(C_k|\mathbf{x})$ = **Posterior** = output keputusan

### Gaussian Naive Bayes (Asumsi Independensi):
$$P(\mathbf{x}|C_k) = \prod_{i=1}^{n} \frac{1}{\sqrt{2\pi\sigma_{ki}^2}} \exp\left(-\frac{(x_i - \mu_{ki})^2}{2\sigma_{ki}^2}\right)$$

**Keputusan**: $\hat{y} = \arg\max_k P(C_k|\mathbf{x})$

### Mengapa Bayesian cocok untuk imbalanced data?
Prior $P(C_k)$ **otomatis memperhitungkan** ketidakseimbangan kelas!

In [ ]:
# ============================================================
# 6.1 — Implementasi Bayesian Decision Classifier Manual
# ============================================================

class BayesianDecisionClassifier:
    """
    Gaussian Naive Bayes Classifier — implementasi manual.
    Tidak menggunakan sklearn sama sekali!
    """
    
    def __init__(self):
        self.classes = None
        self.priors = {}    # P(C_k)
        self.means = {}     # μ per fitur per kelas
        self.variances = {} # σ² per fitur per kelas
    
    def fit(self, X, y):
        """Estimasi parameter dari data latih."""
        self.classes = np.unique(y)
        n_total = len(y)
        
        print('📐 FASE TRAINING — Estimasi Parameter Gaussian:')
        print('=' * 60)
        
        for cls in self.classes:
            mask = (y == cls)
            X_cls = X[mask]
            n_cls = mask.sum()
            
            # Prior: P(C_k) = n_k / n_total
            self.priors[cls] = n_cls / n_total
            
            # Likelihood parameters (Gaussian)
            self.means[cls]     = X_cls.mean(axis=0)
            # Variance + Laplace smoothing kecil
            self.variances[cls] = X_cls.var(axis=0) + 1e-9
            
            cls_name = 'Sehat' if cls == 0 else 'Sakit'
            print(f'\n  Kelas {cls} ({cls_name}) — n={n_cls}, Prior P(C)={self.priors[cls]:.4f}')
            for i, fn in enumerate(top3_features):
                print(f'    {fn:<20}: μ={self.means[cls][i]:.4f}, σ²={self.variances[cls][i]:.6f}')
        
        print('\n✅ Training selesai!')
        return self
    
    def _gaussian_pdf(self, x, mu, var):
        """Hitung Gaussian PDF: P(x | μ, σ²)"""
        coeff = 1.0 / np.sqrt(2 * np.pi * var)
        exponent = np.exp(-((x - mu) ** 2) / (2 * var))
        return coeff * exponent
    
    def predict_proba(self, X):
        """Hitung posterior P(C_k | x) untuk setiap data."""
        posteriors = np.zeros((len(X), len(self.classes)))
        
        for j, cls in enumerate(self.classes):
            # Log-likelihood untuk stabilitas numerik
            log_prior = np.log(self.priors[cls])
            
            log_likelihood = np.zeros(len(X))
            for i in range(X.shape[1]):
                pdf = self._gaussian_pdf(X[:, i], 
                                          self.means[cls][i], 
                                          self.variances[cls][i])
                log_likelihood += np.log(pdf + 1e-300)  # hindari log(0)
            
            posteriors[:, j] = log_prior + log_likelihood
        
        # Softmax untuk normalisasi ke probabilitas [0,1]
        exp_post = np.exp(posteriors - posteriors.max(axis=1, keepdims=True))
        proba = exp_post / exp_post.sum(axis=1, keepdims=True)
        return proba
    
    def predict(self, X):
        """Prediksi kelas: argmax posterior."""
        proba = self.predict_proba(X)
        return self.classes[np.argmax(proba, axis=1)]


# ============================================================
# Train model
# ============================================================
model = BayesianDecisionClassifier()
model.fit(X_selected, y)

In [ ]:
# ============================================================
# 6.2 — Visualisasi Distribusi Gaussian (Likelihood)
# ============================================================
fig, axes = plt.subplots(1, len(top3_features), figsize=(15, 4))
fig.suptitle('📐 Distribusi Gaussian per Kelas — Model Bayesian', fontsize=13, fontweight='bold')

x_plot = np.linspace(0, 1, 300)

for ax, (feat, idx) in zip(axes, [(f, i) for i, f in enumerate(top3_features)]):
    for cls, clr, lbl in [(0, '#4CAF50', 'Sehat'), (1, '#F44336', 'Sakit')]:
        mu  = model.means[cls][idx]
        var = model.variances[cls][idx]
        pdf = (1/np.sqrt(2*np.pi*var)) * np.exp(-((x_plot - mu)**2)/(2*var))
        ax.plot(x_plot, pdf, color=clr, linewidth=2.5, label=f'{lbl} (μ={mu:.3f})')
        ax.fill_between(x_plot, pdf, alpha=0.15, color=clr)
        ax.axvline(mu, color=clr, linestyle='--', linewidth=1.2, alpha=0.7)
        # Scatter titik data asli
        data_pts = X_selected[y == cls, idx]
        ax.scatter(data_pts, np.zeros_like(data_pts) - 0.3,
                   c=clr, marker='|', s=200, linewidth=1.5, zorder=5)
    ax.set_title(f'Fitur: {feat}', fontweight='bold')
    ax.set_xlabel('Nilai Ternormalisasi [0,1]')
    ax.set_ylabel('Densitas Probabilitas')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('09_gaussian_likelihood.png', bbox_inches='tight')
plt.show()
print('💡 Semakin terpisah kurva hijau & merah → fitur semakin diskriminatif!')

In [ ]:
# ============================================================
# 6.3 — Prediksi & Tampilkan Detail per Data
# ============================================================
y_pred = model.predict(X_selected)
y_proba = model.predict_proba(X_selected)

print('🔮 HASIL PREDIKSI — Detail per Data:')
print('=' * 80)
print(f'{"No":>3} {"Label Asli":>12} {"Prediksi":>12} {"P(Sehat)":>10} {"P(Sakit)":>10} {"Status":>12}')
print('-' * 80)
for i in range(len(y)):
    true_name = 'Sehat' if y[i] == 0 else 'Sakit'
    pred_name = 'Sehat' if y_pred[i] == 0 else 'Sakit'
    status = '✅ BENAR' if y[i] == y_pred[i] else '❌ SALAH'
    print(f'{i+1:>3} {true_name:>12} {pred_name:>12} {y_proba[i,0]:>10.4f} {y_proba[i,1]:>10.4f} {status:>12}')
print('=' * 80)
acc = (y == y_pred).mean()
print(f'\n📊 Accuracy keseluruhan: {acc:.2%}')

---
# 📈 STEP 7 — Evaluasi: Precision, Recall, F1-Score (Manual)

Untuk dataset imbalanced, **Accuracy saja TIDAK cukup!**

| Metrik | Rumus | Makna |
|---|---|---|
| **Precision** | $\frac{TP}{TP+FP}$ | Dari semua yang diprediksi Sakit, berapa % yang benar-benar sakit? |
| **Recall** | $\frac{TP}{TP+FN}$ | Dari semua yang benar-benar Sakit, berapa % yang berhasil terdeteksi? |
| **F1-Score** | $\frac{2 \cdot P \cdot R}{P + R}$ | Harmonic mean Precision & Recall |

**Untuk kasus medis: Recall lebih penting!** (jangan sampai pasien sakit tidak terdeteksi)

In [ ]:
# ============================================================
# 7.1 — Confusion Matrix Manual
# ============================================================
TP = int(((y == 1) & (y_pred == 1)).sum())
TN = int(((y == 0) & (y_pred == 0)).sum())
FP = int(((y == 0) & (y_pred == 1)).sum())
FN = int(((y == 1) & (y_pred == 0)).sum())

print('🧮 CONFUSION MATRIX (Manual):')
print('=' * 45)
print('                  Prediksi')
print('                  Sehat(0)  Sakit(1)')
print(f'  Asli  Sehat(0)  TN={TN:>4}    FP={FP:>4}')
print(f'        Sakit(1)  FN={FN:>4}    TP={TP:>4}')
print('=' * 45)
print(f'\n  TP (True Positive)  = {TP} → Sakit & diprediksi Sakit')
print(f'  TN (True Negative)  = {TN} → Sehat & diprediksi Sehat')
print(f'  FP (False Positive) = {FP} → Sehat tapi diprediksi Sakit (Type I Error)')
print(f'  FN (False Negative) = {FN} → Sakit tapi diprediksi Sehat (Type II Error) ⚠️')

In [ ]:
# ============================================================
# 7.2 — Hitung Precision, Recall, F1 Manual
# ============================================================
precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
f1        = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
accuracy  = (TP + TN) / (TP + TN + FP + FN)

# Untuk kelas 0 (Sehat)
precision_0 = TN / (TN + FN) if (TN + FN) > 0 else 0.0
recall_0    = TN / (TN + FP) if (TN + FP) > 0 else 0.0
f1_0        = (2 * precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0.0

# Macro average
precision_macro = (precision + precision_0) / 2
recall_macro    = (recall + recall_0) / 2
f1_macro        = (f1 + f1_0) / 2

print('📊 LAPORAN EVALUASI LENGKAP')
print('=' * 60)
print(f'{"":>20} {"Precision":>12} {"Recall":>10} {"F1-Score":>10} {"Support":>10}')
print('-' * 60)
print(f'{"Sehat (0)":>20} {precision_0:>12.4f} {recall_0:>10.4f} {f1_0:>10.4f} {int((y==0).sum()):>10}')
print(f'{"Sakit (1)":>20} {precision:>12.4f} {recall:>10.4f} {f1:>10.4f} {int((y==1).sum()):>10}')
print('-' * 60)
print(f'{"Macro Avg":>20} {precision_macro:>12.4f} {recall_macro:>10.4f} {f1_macro:>10.4f} {len(y):>10}')
print('=' * 60)
print(f'\n  ✅ Accuracy  : {accuracy:.4f} ({accuracy:.2%})')
print(f'  🎯 Precision : {precision:.4f} — Ketepatan deteksi Sakit')
print(f'  📡 Recall    : {recall:.4f} — Kelengkapan deteksi Sakit')
print(f'  ⚖️  F1-Score  : {f1:.4f} — Keseimbangan P & R')

In [ ]:
# ============================================================
# 7.3 — Visualisasi Confusion Matrix (Visual)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('📈 Evaluasi Model Bayesian Decision Classifier', fontsize=13, fontweight='bold')

# --- Confusion Matrix ---
cm = np.array([[TN, FP], [FN, TP]])
ax = axes[0]
im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=max(TN, TP, FP+1, FN+1)+1)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Prediksi Sehat (0)', 'Prediksi Sakit (1)'], fontsize=10)
ax.set_yticklabels(['Asli Sehat (0)', 'Asli Sakit (1)'], fontsize=10)
ax.set_xlabel('Prediksi', fontsize=11, fontweight='bold')
ax.set_ylabel('Label Asli', fontsize=11, fontweight='bold')
ax.set_title('Confusion Matrix', fontweight='bold', fontsize=11)

labels_cm = [['TN', 'FP'], ['FN', 'TP']]
for i in range(2):
    for j in range(2):
        val = cm[i, j]
        color = 'white' if val > cm.max()/2 else 'black'
        ax.text(j, i, f'{labels_cm[i][j]}\n{val}',
                ha='center', va='center', fontsize=14, fontweight='bold', color=color)

# Border merah untuk FN (paling berbahaya)
rect = plt.Rectangle((-0.5 + 0, 0.5), 1, 1, linewidth=3, edgecolor='red', facecolor='none')
ax.add_patch(rect)
ax.text(0, 1.45, '⚠️ FN berbahaya!', ha='center', fontsize=8, color='red', fontweight='bold')

# --- Metrik Bar Chart ---
ax2 = axes[1]
metrik_names  = ['Accuracy', 'Precision\n(Sakit)', 'Recall\n(Sakit)', 'F1-Score\n(Sakit)', 'F1-Macro']
metrik_values = [accuracy, precision, recall, f1, f1_macro]
bar_colors3 = ['#2196F3', '#FF9800', '#E91E63', '#9C27B0', '#607D8B']

bars = ax2.bar(metrik_names, metrik_values, color=bar_colors3, edgecolor='black', width=0.55)
ax2.set_ylim(0, 1.15)
ax2.set_title('Semua Metrik Evaluasi', fontweight='bold', fontsize=11)
ax2.set_ylabel('Skor (0 – 1)')
ax2.axhline(1.0, color='green', linestyle='--', alpha=0.4, linewidth=1)
ax2.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, metrik_values):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.02,
             f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('10_evaluation_metrics.png', bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# 7.4 — Visualisasi Probabilitas Prediksi per Data
# ============================================================
fig, ax = plt.subplots(figsize=(13, 5))
fig.suptitle('🔮 Probabilitas Posterior P(Sakit|x) per Data', fontsize=13, fontweight='bold')

x_idx = np.arange(len(y))
proba_sakit = y_proba[:, 1]

# Bar probabilitas
bar_clrs = ['#F44336' if p >= 0.5 else '#4CAF50' for p in proba_sakit]
bars = ax.bar(x_idx, proba_sakit, color=bar_clrs, edgecolor='black', alpha=0.8, width=0.6)

# Garis decision boundary
ax.axhline(0.5, color='black', linestyle='--', linewidth=2, label='Decision Boundary (0.5)')

# Tandai label asli
for i, (true_label, prob) in enumerate(zip(y, proba_sakit)):
    marker = '❤️' if true_label == 1 else ''
    if true_label == 1:
        ax.text(i, prob + 0.03, '🩺', ha='center', fontsize=12)
    correct = '✓' if (prob >= 0.5) == true_label else '✗'
    clr = 'green' if correct == '✓' else 'red'
    ax.text(i, -0.06, correct, ha='center', fontsize=11, color=clr, fontweight='bold')

ax.set_xticks(x_idx)
ax.set_xticklabels([f'D{i+1}\n({"S" if y[i]==1 else "H"})' for i in range(len(y))], fontsize=8)
ax.set_ylabel('P(Sakit | x)', fontsize=11)
ax.set_xlabel('Data ke-n  (H=Sehat, S=Sakit — label asli)', fontsize=10)
ax.set_ylim(-0.12, 1.15)
ax.grid(axis='y', alpha=0.3)

red_patch  = mpatches.Patch(color='#F44336', label='Prediksi: Sakit')
green_patch = mpatches.Patch(color='#4CAF50', label='Prediksi: Sehat')
ax.legend(handles=[red_patch, green_patch,
                   mpatches.Patch(color='none', label='🩺 = Data Sakit Asli')],
          fontsize=9, loc='upper left')

plt.tight_layout()
plt.savefig('11_prediction_probabilities.png', bbox_inches='tight')
plt.show()
print('💡 ✓ = prediksi benar, ✗ = prediksi salah. 🩺 = data yang memang sakit.')

---
# 🗺️ STEP 8 — Decision Boundary Visualization

Visualisasikan **batas keputusan** model di ruang 2 fitur terbaik.

In [ ]:
# ============================================================
# 8 — Decision Boundary (2 fitur terbaik)
# ============================================================
# Gunakan 2 fitur dengan fisher score tertinggi
top2 = sorted(fisher_scores.items(), key=lambda x: x[1], reverse=True)[:2]
f1_name, f2_name = top2[0][0], top2[1][0]
f1_idx = features.index(f1_name)
f2_idx = features.index(f2_name)

X_2d = X_norm[:, [f1_idx, f2_idx]]

# Latih model 2D
model_2d = BayesianDecisionClassifier()

# Redirect print
import io, sys
old_stdout = sys.stdout; sys.stdout = io.StringIO()
model_2d.fit(X_2d, y)
sys.stdout = old_stdout

# Grid prediksi
h = 0.01
xx, yy = np.meshgrid(np.arange(-0.1, 1.15, h), np.arange(-0.1, 1.15, h))
grid = np.c_[xx.ravel(), yy.ravel()]
Z = model_2d.predict(grid).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(9, 7))
fig.suptitle('🗺️ Decision Boundary — Bayesian Classifier', fontsize=14, fontweight='bold')

ax.contourf(xx, yy, Z, alpha=0.25, cmap=plt.cm.RdYlGn)
ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2, linestyles='--')

for cls, clr, mrk, lbl in [(0, '#2E7D32', 'o', 'Sehat (0)'), (1, '#C62828', '^', 'Sakit (1)')]:
    mask = y == cls
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=clr, marker=mrk, s=130, edgecolors='black', linewidths=1.2,
               label=lbl, zorder=5)

# Tandai kesalahan prediksi
y_pred_2d = model_2d.predict(X_2d)
wrong_mask = y != y_pred_2d
if wrong_mask.sum() > 0:
    ax.scatter(X_2d[wrong_mask, 0], X_2d[wrong_mask, 1],
               s=300, facecolors='none', edgecolors='blue', linewidths=2.5,
               label='Prediksi Salah', zorder=6)

ax.set_xlabel(f'{f1_name} (ternormalisasi)', fontsize=11)
ax.set_ylabel(f'{f2_name} (ternormalisasi)', fontsize=11)
ax.set_xlim(-0.05, 1.1)
ax.set_ylim(-0.05, 1.1)
ax.legend(fontsize=10, loc='upper left')
ax.grid(alpha=0.3)

# Teks region
ax.text(0.05, 0.92, 'Region: Sakit', transform=ax.transAxes,
        color='red', fontsize=10, fontweight='bold', alpha=0.7)
ax.text(0.65, 0.08, 'Region: Sehat', transform=ax.transAxes,
        color='green', fontsize=10, fontweight='bold', alpha=0.7)

plt.tight_layout()
plt.savefig('12_decision_boundary.png', bbox_inches='tight')
plt.show()
print(f'💡 Garis putus-putus = batas keputusan. Kiri atas = zona Sakit, Kanan bawah = zona Sehat.')
print(f'   Lingkaran biru = data yang diprediksi salah.')

---
# 📋 STEP 9 — Ringkasan Pipeline & Kesimpulan

In [ ]:
# ============================================================
# 9 — Ringkasan Visual Pipeline
# ============================================================
fig = plt.figure(figsize=(16, 4))
fig.suptitle('🗺️ Ringkasan Pipeline Pengenalan Pola', fontsize=14, fontweight='bold', y=1.02)

steps = [
    ('1\nDataset\nDummy', '#E3F2FD', '#1565C0'),
    ('2\nEDA &\nVisualisasi', '#E8F5E9', '#1B5E20'),
    ('3\nData\nCleaning', '#FFF9C4', '#F57F17'),
    ('4\nNormalisasi\nMin-Max', '#FCE4EC', '#880E4F'),
    ('5\nFeature\nSelection', '#EDE7F6', '#4527A0'),
    ('6\nBayesian\nClassifier', '#E0F2F1', '#004D40'),
    ('7\nEvaluasi\nP/R/F1', '#FBE9E7', '#BF360C'),
]

ax = fig.add_subplot(111)
ax.axis('off')

n = len(steps)
w, h_box = 1.2, 0.7
gap = 0.4

for i, (label, facecolor, textcolor) in enumerate(steps):
    x = i * (w + gap)
    rect = mpatches.FancyBboxPatch((x, 0.1), w, h_box,
                                    boxstyle='round,pad=0.05',
                                    facecolor=facecolor, edgecolor=textcolor, linewidth=2)
    ax.add_patch(rect)
    ax.text(x + w/2, 0.1 + h_box/2, label,
            ha='center', va='center', fontsize=9.5,
            fontweight='bold', color=textcolor)
    # Panah
    if i < n - 1:
        ax.annotate('', xy=(x + w + gap, 0.45), xytext=(x + w + 0.03, 0.45),
                    arrowprops=dict(arrowstyle='->', color='gray', lw=1.8))

ax.set_xlim(-0.2, n * (w + gap))
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('13_pipeline_summary.png', bbox_inches='tight')
plt.show()

print('\n' + '='*65)
print('  📋 KESIMPULAN PIPELINE PENGENALAN POLA')
print('='*65)
print(f'  Dataset      : 20 data, 4 fitur, extremely imbalanced (85:15)')
print(f'  Cleaning     : Imputasi median per kelas (2 missing values)')
print(f'  Normalisasi  : Min-Max Scaling → [0, 1]')
print(f'  Fitur Pilihan: {top3_features}')
print(f'  Model        : Bayesian Decision (Gaussian Naive Bayes manual)')
print(f'  Accuracy     : {accuracy:.2%}')
print(f'  Precision    : {precision:.4f}  (kelas Sakit)')
print(f'  Recall       : {recall:.4f}  (kelas Sakit)')
print(f'  F1-Score     : {f1:.4f}  (kelas Sakit)')
print('='*65)
print()
print('  💡 INSIGHT UTAMA:')
print('  1. Accuracy tinggi bisa MENYESATKAN pada data imbalanced!')
print('     → Model bisa benar 85% hanya dengan selalu prediksi "Sehat"')
print('  2. Recall kelas Sakit = metrik PALING PENTING di domain medis')
print('     → FN (missed sick) lebih berbahaya dari FP')
print('  3. Bayesian Decision menggunakan prior P(C) secara otomatis')
print('     → Prior yang kecil untuk kelas Sakit membuat model bias ke Sehat')
print('  4. Feature Selection membantu mengurangi noise dan mempercepat')
print('     komputasi tanpa kehilangan informasi penting')
print('='*65)

---

## 🎓 Latihan Mandiri

Coba modifikasi notebook ini untuk eksplorasi lebih lanjut:

1. **Ubah threshold** dari 0.5 menjadi 0.3 → bagaimana Recall berubah?
2. **Tambah fitur** `resting_ecg` atau `chest_pain_type` → apakah F1 meningkat?
3. **Coba hanya 1 fitur** terbaik → seberapa jauh penurunan performa?
4. **Ubah distribusi imbalance** menjadi 90:10 → apa yang terjadi pada Recall?
5. **Bandingkan** dengan model sederhana lain: Nearest Centroid (implementasi manual)

---

**© Mata Kuliah Pengenalan Pola** | Pipeline dibuat sepenuhnya secara manual tanpa sklearn untuk tujuan pembelajaran